In [ ]:
%pip install numpy matplotlib qiskit qiskit-aer scipy --quiet

In [ ]:
from __future__ import annotations
import math
from dataclasses import dataclass, field
from typing import Callable, Sequence
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import RYGate, RZGate, MCXGate, QFT, QFTGate
from qiskit.quantum_info import Statevector, Operator
import Layer2
import Layer3

In [ ]:
#Re-usable subroutines for the application test layer

def select_oracle(U:Sequence[QuantumCircuit], tgt: int) -> QuantumCircuit: #Builds a sequential controlled application of unitaries
    size = len(U)
    k = int(math.ceil(math.log2(max(size, 2))))
    select = QuantumRegister(k)
    target = QuantumRegister(tgt)
    qc = QuantumCircuit(select, target, name="SELECT")

    for i, u in enumerate(U):
        bits = [(i >> j) & 1 for j in range(k)]
        for j, b in enumerate(bits):
            if b == 0:
                qc.x(sel[j])
        cntrld_U = U.to_gate().control(k)
        qc.append(cntrld_U, list(select)+list(target))
        for j, b in enumerate(bits):
            if b == 0:
                qc.x(sel[j])
    return qc

In [ ]:
#Amplitude amplification

def grover(state_prep:QuantumCircuit, oracle:QuantumCircuit) -> QuantumCircuit:
    n = state_prep.num_qubits
    qr = QuantumRegister(n)
    qc = QuantumCircuit(qr)

    qc.compose(oracle, qubits=qr, inplace=True)
    qc.compose(state_prep.inverse(), qubits=qr, inplace=True)

    for q in qr:
        qc.x(q)
    qc.h(qr[-1])
    if n > 1:
        qc.append(MCXGate(n - 1), list(qr))
    else:
        qc.z(qr[-1])
    qc.h(qr[-1])
    for q in qr:
        qc.x(q)
    qc.global_phase += math.pi
    qc.compose(state_prep, qubits=qr, inplace=True)

    return qc

def amp_amplify(state_prep:QuantumCircuit, oracle:QuantumCircuit, iters:int) -> QuantumCircuit:
    qc = state_prep.copy()
    Q = grover(state_prep, oracle)
    for _ in range(iters):
        qc.compose(Q, inplace=True)

    return qc

def optimal_iters(amp:float) -> int:
    theta = math.asin(min(1.0, max(-1.0,amp)))
    if theta < 1e-12:
        return 0

    return max(0, int(round(math.pi/(4.0*theta) - 0.5)))

In [ ]:
#Iterative amplitude estimation

@dataclass
class amp_est_res:
    amp_est: float
    confidence_interval: tuple[float, float]
    total_queries: int
    rounds: int

def _mark_oracle(oracle:QuantumCircuit, qubits:int) -> list[bool]: #Applies the oracle to each computational basis state and flags those with negative real amplitude or any imaginary component
    op = Operator(oracle)
    marked = []
    for i in range(2**qubits):
        phi = op.data[i, i]
        marked.append(abs(phi + 1.0) < 1e-6)
    return marked

def AMP_EST(state_prep:QuantumCircuit, oracle:QuantumCircuit, eps:float, alpha:float, max_rounds:int, shots:int, rng:np.random.Generator | None = None) -> amp_est_res:
    n = state_prep.num_qubits
    sv = Statevector.from_instruction(state_prep)
    marked = _mark_oracle(oracle, n)
    a_true = float(sum(abs(sv.data[i])**2 for i in range(2**n) if marked[i]))
    theta_true = math.asin(math.sqrt(a_true))

    theta_low, theta_high = 0.0, math.pi/2.0   #Theta interval
    k = 0
    upper_half = False  #Tracks which half of the target quadrant we sit in
    total_queries = 0
    rounds = 0

    for rounds in range(1, max_rounds + 1):
        width = theta_high - theta_low
        if width <= 2*eps:
            break
        K_prev = 2*k+1   #Find the largest K
        K = K_prev
        while True:
            K_next = K+2
            low_frac = (K_next*theta_low)/(math.pi/2.0)
            high_frac = (K_next*theta_high)/(math.pi/2.0)
            if math.floor(low_frac) != math.floor(high_frac):   #Both the endpoints must fall in the same quadrant
                break
            K = K_next

        k = (K-1)//2

        q_idx = int(math.floor(K*0.5*(theta_low + theta_high)/(math.pi/2.0)))   #Determine the direction from the current quadrant index
        upper_half = (q_idx%2 == 1)

        p_true = math.sin(K*theta_true)**2
        scss = rng.binomial(shots, p_true)
        ratio = scss/shots
        delta = math.sqrt(math.log(2.0/alpha)/(2*shots))
        ratio_low = max(0.0, ratio-delta)
        ratio_high = min(1.0, ratio+delta)

        alpha_q = q_idx*math.pi/2.0   #Invert sin^2 to get angle in the correct quadrant
        if not upper_half:            #Checking for quadrant
            angle_low = math.asin(math.sqrt(ratio_low))
            angle_high = math.asin(math.sqrt(ratio_high))
        else:
            angle_low = math.pi/2.0 - math.asin(math.sqrt(ratio_high))
            angle_high = math.pi/2.0 - math.asin(math.sqrt(ratio_low))

        new_theta_low = (alpha_q + angle_low)/K
        new_theta_high = (alpha_q + angle_high)/K
        theta_low = max(theta_low, new_theta_low)   #Intersection
        theta_high = min(theta_high, new_theta_high)
        if theta_high < theta_low:
            theta_high = theta_low
        total_queries+=shots*K

    amp_low = math.sin(theta_low)**2
    amp_high = math.sin(theta_high)**2
    amp_est = 0.5 * (amp_low + amp_high)

    return amp_est_res(amp_est=amp_est, confidence_interval=(amp_low, amp_high), total_queries=total_queries, rounds=rounds)

In [ ]:
#Linear Combination of Unitaries(LCU) block encoding

def lcu_block_encoding(coeffs:np.ndarray, U:Sequence[QuantumCircuit]) -> QuantumCircuit:
    L = len(U)
    k = int(math.ceil(math.log2(max(L,2))))
    tgt = U[0].num_qubits
    select = QuantumRegister(k)
    target = QuantumRegister(tgt)
    qc = QuantumCircuit(select, target, name="LCU")

    from layer3 import gr_circuit, _reverse_bit
    alphas = np.zeros(2**k)
    alphas[:L] = np.asarray(coeffs)
    total = float(alphas.sum())
    if total < 1e-15:
        return qc

    probs = alphas/total
    prep = gr_circuit(probs[_reverse_bit(k)])

    qc.compose(prep, qubits=list(select), inplace=True)
    qc.compose(select_oracle(list(U), n_tgt), qubits=list(select) + list(target), inplace=True)
    qc.compose(prep.inverse(), qubits=list(select), inplace=True)

    return qc

In [ ]:
#Qubitization walk (instead of Trotterization)

# def qubitization_walk(block:QuantumCircuit, signal_qubits:int) -> QuantumCircuit:
#     qr = QuantumRegister(block.num_qubits)
#     qc = QuantumCircuit(qr)
#     qc.compose(block, inplace=True)

#     for i in range(signal_qubits):   #Reflection on the ancilla register
#         qc.x(qr[i])

#     qc.h(qr[signal_qubits-1])
#     if signal_qubits > 1:
#         qc.append(MCXGate(signal_qubits-1), list(qr[:signal_qubits]))
#     else:
#         qc.z(qr[0])

#     qc.h(qr[signal_qubits-1])
#     for i in range(signal_qubits):
#         qc.x(qr[i])

#     return qc

In [ ]:
#QSP phase sequence

# def qsp_phase_circuit(phases:Sequence[float], walk_op:QuantumCircuit) -> QuantumCircuit:   #Applies a phase to the block encoding
#     n = walk_op.num_qubits
#     qr = QuantumRegister(n)
#     qc = QuantumCircuit(qr, name="QSP")
#     qc.rz(2.0*phases[0], qr[0])   #Initial phase
#     for phi in phases[1:]:
#         qc.compose(walk_op, inplace=True)
#         qc.rz(2.0*phi, qr[0])
#     return qc

# def qsvt_angles_from_polynomial(coeffs: np.ndarray) -> list[float]:
#     return [0.0] * len(coeffs)

In [ ]:
def _phase_oracle(n:int, marked:int) -> QuantumCircuit:  #Phase oracle that applies -1 to a single marked computational state
    qr = QuantumRegister(n)
    qc = QuantumCircuit(qr)

    bits = [(marked >> i) & 1 for i in range(n)]
    for j, b in enumerate(bits):
        if b == 0:
            qc.x(qr[j])

    qc.h(qr[-1])
    if n > 1:
        qc.append(MCXGate(n-1), list(qr))
    else:
        qc.z(qr[0])

    qc.h(qr[-1])
    for k, b in enumerate(bits):
        if b == 0:
            qc.x(qr[k])

    return qc

def _uniform_prep(n:int) -> QuantumCircuit:
    qr = QuantumRegister(n)
    qc = QuantumCircuit(qr)
    for q in qr:
        qc.h(q)

    return qc

In [ ]:
#Small demo

def _demo_layer5() -> None:
    print("Layer 5")

    n = 3   #Amplitude amplification
    marked = 5
    prep = _uniform_prep(n)
    oracle = _phase_oracle(n, marked)

    a_init = math.sqrt(1.0/2**n)    #Initial amplitude
    k_opt = optimal_iters(a_init)
    print(f"Amplitude Amplification, marked at: |{marked:0{n}b}>:")
    print(f"Initial amplitude: {a_init:.4f}")
    print(f"Optimal iterations: {k_opt}")

    aa_circ = amp_amplify(prep, oracle, k_opt)
    sv = Statevector.from_instruction(aa_circ)
    p_marked = abs(sv.data[marked])**2
    print(f"Post amplification: {p_marked:.4f}")

    result = AMP_EST(prep, oracle, epsilon=0.01, alpha=0.05, shots_per_round=500, rng=np.random.default_rng(42))  #Iterative amplitude estimation on same setup
    print(f"Amplitude Estimation: {result.amp_est:.4f}")
    print(f"Confidence Interval[{result.confidence_interval[0]:.4f}, {result.confidence_interval[1]:.4f}]")
    print(f"Rounds: {result.rounds}")
    print(f"Total queries to the oracle: {result.total_queries}")

    U_I = QuantumCircuit(1, name="I")  #LCU
    U_X = QuantumCircuit(1, name="X")
    lcu = lcu_block_encoding(np.array([1.0, 1.0]), [U_I, U_X])
    print(f"LCU block encoding of (I+X)/2: {lcu.num_qubits} qubits, depth: {lcu.depth()} and size: {lcu.size()}")

In [ ]:
if __name__ == "__main__":
    _demo_layer5()